# Predictive Analytics Baseline Model

## 1. Modelling Plan
The objective is to predict whether a client will subscribe to a term deposit (column `y`). This is a binary classification problem.
We use a Logistic Regression model as a baseline because it is interpretable, robust, fast to train, and sets a strong fundamental baseline against which more complex models (like Random Forests or Gradient Boosted Trees) can be compared.

## 2. Prediction Type & Evaluation Metrics
- **Prediction Type**: Binary Classification.
- **Evaluation Metric**: **PR-AUC (Average Precision)** is chosen as the primary evaluation metric because the target variable is highly imbalanced (most people do not subscribe). We also report ROC-AUC and the Confusion Matrix to understand false positives and false negatives.

## 3. Train/Test Split
We will use an 80/20 train/test split. A stratified split is used to ensure the train and test distributions of the target class remain identical, maintaining statistical validity. A random seed (`random_state=42`) guarantees reproducibility. We also perform 5-fold cross-validation on the train split to evaluate variance.

In [1]:
import pandas as pd
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_validate
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, roc_auc_score, average_precision_score, confusion_matrix

# Load Data
df = pd.read_csv('bank-additional-full.csv', sep=';')

## 5. Preprocessing Steps & Artefact Handling
# Drop 'duration' feature. As specified by the dataset creators: 
# 'this attribute highly affects the output target... Yet, the duration is not known before a call is performed.'
df = df.drop(columns=['duration'])

# LEAKAGE/ARTEFACT FIX: 'pdays' uses 999 to denote "not previously contacted".
# Feeding 999 into a linear model heavily distorts the mathematical weights.
df['contacted_previously'] = (df['pdays'] != 999).astype(int)
df = df.drop(columns=['pdays'])

# Convert target to binary
df['y'] = df['y'].map({'no': 0, 'yes': 1})

# Identify feature types
categorical_features = df.select_dtypes(include=['object']).columns.tolist()
numerical_features = df.select_dtypes(exclude=['object', 'datetime']).columns.tolist()
if 'y' in numerical_features: numerical_features.remove('y')
if 'contacted_previously' in numerical_features: numerical_features.remove('contacted_previously')
numerical_features.append('contacted_previously')

# Define X and y
X = df.drop(columns=['y'])
y = df['y']

## 3. Reproducible Train/Test split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# Create preprocessing pipelines
preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), numerical_features),
        ('cat', OneHotEncoder(handle_unknown='ignore', drop='if_binary'), categorical_features)
    ])

In [2]:
## 4. Build a Reasonable Baseline Model
# Balanced classes to penalise missing the rare 'yes' target class
baseline_model = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('classifier', LogisticRegression(max_iter=1000, random_state=42, class_weight='balanced'))
])

# Robust Training & Cross-Validation Evaluation
print("Running 5-Fold Stratified Cross-Validation on Train Data...")
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
cv_results = cross_validate(
    baseline_model, X_train, y_train, cv=cv, 
    scoring=['roc_auc', 'average_precision'],
    n_jobs=-1
)
print(f"CV ROC-AUC Mean: {cv_results['test_roc_auc'].mean():.4f}")
print(f"CV PR-AUC Mean: {cv_results['test_average_precision'].mean():.4f}")

## 6. Report Evaluation Results on Hold-out Test Set
baseline_model.fit(X_train, y_train)

y_pred = baseline_model.predict(X_test)
y_pred_proba = baseline_model.predict_proba(X_test)[:, 1]

print("\n--- HOLD-OUT TEST SET EVALUATION ---")
print("Classification Report:")
print(classification_report(y_test, y_pred))

roc_auc_test = roc_auc_score(y_test, y_pred_proba)
pr_auc_test = average_precision_score(y_test, y_pred_proba)

print(f"Test ROC-AUC: {roc_auc_test:.4f}")
print(f"Test PR-AUC: {pr_auc_test:.4f}")

cm = confusion_matrix(y_test, y_pred)
print(f"\nConfusion Matrix:\n{cm}")

Running 5-Fold Stratified Cross-Validation on Train Data...
CV ROC-AUC Mean: 0.7896
CV PR-AUC Mean: 0.4438

--- HOLD-OUT TEST SET EVALUATION ---
Classification Report:
              precision    recall  f1-score   support

           0       0.95      0.86      0.90      7310
           1       0.37      0.65      0.47       928

    accuracy                           0.84      8238
   macro avg       0.66      0.75      0.69      8238
weighted avg       0.88      0.84      0.85      8238

Test ROC-AUC: 0.8010
Test PR-AUC: 0.4601

Confusion Matrix:
[[6280 1030]
 [ 329  599]]


## 7. Baseline Justification
This Logistic Regression model acts as a suitable baseline for several reasons:
- **Simplicity & Interpretability**: We avoid over-engineering while establishing a benchmark.
- **Mitigates Data Leakage**: The highly leaky `duration` feature was omitted to ensure strict, real-world predictability. We also properly resolved the mathematically distortionary `pdays=999` artefact.
- **Imbalance Handling**: The algorithm leverages balanced class weights (as standard accuracy might be misleading). We use PR-AUC for a stringent evaluation on the minority class.
- **Robustness**: 5-Fold Stratified CV ensures the performance is consistent before we finalise on the holdout test set.